# 00 — Data Validation

**Goal:** Prove the experiment is valid before any modeling. Most candidates skip this; experimentation teams live on it.

Checks:
1. Sample Ratio Mismatch (SRM) — a failed SRM invalidates everything downstream
2. Covariate balance — T ⊥ X by design; |SMD| should be ≈ 0 for all features
3. Naive ATE — the ground-truth anchor every CATE model must average back to
4. Leakage scan — confirm `exposure` is excluded and no feature predicts T perfectly

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats
from src.data import load_data, FEATURE_COLS

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

In [ ]:
df = load_data(percent10=True)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nDtype summary:')
df.dtypes.value_counts()

## 1. Sample Ratio Mismatch (SRM)

In [ ]:
n_total = len(df)
n_treated = (df['treatment'] == 1).sum()
n_control = (df['treatment'] == 0).sum()
observed_ratio = n_treated / n_total

# Expected: ~85% treated per dataset documentation
EXPECTED_TREAT_RATIO = 0.85
expected_t = int(n_total * EXPECTED_TREAT_RATIO)
expected_c = n_total - expected_t

chi2, p_val = stats.chisquare(
    f_obs=[n_treated, n_control],
    f_exp=[expected_t, expected_c]
)

print('=== Sample Ratio Mismatch Check ===')
print(f'Observed  treated: {n_treated:,}  ({100*n_treated/n_total:.2f}%)')
print(f'Observed  control: {n_control:,}  ({100*n_control/n_total:.2f}%)')
print(f'Expected  treated: {expected_t:,}  ({100*EXPECTED_TREAT_RATIO:.2f}%)')
print(f'Chi-square statistic: {chi2:.2f},  p-value: {p_val:.4f}')
print()
if p_val < 0.05:
    print('WARNING: SRM detected (p < 0.05). Investigate before proceeding.')
else:
    print('PASS: No SRM detected. Treatment ratio is consistent with expectation.')

## 2. Covariate Balance (Standardized Mean Differences)

In [ ]:
treated = df[df['treatment'] == 1]
control = df[df['treatment'] == 0]

rows = []
for col in FEATURE_COLS:
    m_t = treated[col].mean()
    m_c = control[col].mean()
    s_t = treated[col].std()
    s_c = control[col].std()
    n_t = len(treated)
    n_c = len(control)
    pooled_sd = np.sqrt(((n_t - 1) * s_t**2 + (n_c - 1) * s_c**2) / (n_t + n_c - 2))
    smd = (m_t - m_c) / pooled_sd if pooled_sd > 0 else 0
    rows.append({'feature': col, 'mean_treated': m_t, 'mean_control': m_c, 'SMD': smd})

balance = pd.DataFrame(rows).set_index('feature')
imbalanced = balance[balance['SMD'].abs() > 0.1]

print('Covariate balance (|SMD| > 0.1 = potentially imbalanced):')
print(balance.round(4).to_string())
print(f'\nFeatures with |SMD| > 0.1: {list(imbalanced.index)}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#d62728' if abs(v) > 0.1 else '#1f77b4' for v in balance['SMD']]
ax.barh(balance.index, balance['SMD'].abs(), color=colors)
ax.axvline(0.1, color='red', linestyle='--', linewidth=1.2, label='|SMD|=0.1 threshold')
ax.set_xlabel('|Standardized Mean Difference|')
ax.set_title('Covariate Balance Check (T ⊥ X by randomization design)')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/covariate_balance.png', bbox_inches='tight')
plt.show()

## 3. Naive ATE with 95% CI

In [ ]:
for label in ['visit', 'conversion']:
    if label not in df.columns:
        continue
    y1 = treated[label].values
    y0 = control[label].values
    ate = y1.mean() - y0.mean()
    se = np.sqrt(y1.var()/len(y1) + y0.var()/len(y0))
    ci_lo, ci_hi = ate - 1.96*se, ate + 1.96*se
    # two-proportion z-test
    z = ate / se
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    print(f'--- {label} ---')
    print(f'  Treated rate:  {y1.mean():.4%}')
    print(f'  Control rate:  {y0.mean():.4%}')
    print(f'  ATE:           {ate:.4%}  [{ci_lo:.4%}, {ci_hi:.4%}]')
    print(f'  z={z:.2f},  p={p:.2e}')
    print()

## 4. Leakage Scan

In [ ]:
# Confirm exposure is excluded
print('exposure in columns:', 'exposure' in df.columns)
print('Reason: exposure is post-treatment; conditioning on it induces collider bias.')
print()

# Check if any feature perfectly predicts treatment
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X = df[FEATURE_COLS].values
t = df['treatment'].values

lr = LogisticRegression(max_iter=200, random_state=42)
lr.fit(X, t)
auc = roc_auc_score(t, lr.predict_proba(X)[:, 1])
print(f'Propensity model AUC (logistic on all features): {auc:.4f}')
print('If AUC >> 0.5, features predict treatment — randomization issue.')
if auc > 0.6:
    print('WARNING: features have non-trivial predictive power over T.')
else:
    print('PASS: features do not strongly predict T. Randomization appears clean.')

## Summary

| Check | Result |
|---|---|
| SRM (p-value) | See above |
| Imbalanced features (\|SMD\|>0.1) | See balance table |
| ATE (visit) | See above |
| Leakage / propensity AUC | See above |

**Key insight:** Because T is randomized, identification is clean. We are solving a pure *estimation* problem (modeling heterogeneity in τ(x)), not an identification problem (removing confounding). This distinction — which most candidates blur — is what makes the Criteo dataset a good benchmark for CATE methods.